# Introduction

The goals of this analysis is to train, predict, evaluate, and SHAP BPNet models. 

We will be training second-pass models including: ATAC residual model to learn accessibility grammar on:

+ genome-wide
+ 50/50 split accessible regions
+ accessible regions only

# Computational setup

In [2]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import warnings
warnings.filterwarnings("ignore")
from tensorflow.python.util import deprecation
deprecation._PRINT_DEPRECATION_WARNINGS = False

#Packages
import os
import sys
import keras
import pyBigWig
import json
import pandas as pd
import numpy as np
import keras.backend as K
from keras.models import load_model
from tqdm import tqdm

import plotnine
from plotnine import *

# Settings

## Working options
working_dir = f'/n/projects/mw2098/publications/2024_weilert_acc/code/2_analysis/'
os.chdir(working_dir)
pd.set_option('display.max_columns', 100)
figure_path = 'figures/3c_train_differentiation_combined'
bpreveal_path = '/n/projects/mw2098/publications/2024_weilert_acc/public/software/bpreveal_404/'
python_path = '/home/mw2098/anaconda3/envs/bpreveal_404/bin/python'

## Custom functions
sys.path.insert(0, f'scripts/py/functions')
from metrics import compute_auprc
from functional import one_hot_encode_sequences, one_hot_encode_sequence, one_hot_decode_sequence, shuffle_seqs, logitsToProfile, insert_motif
from perturb import generate_random_seq
from motifs import extract_seqs_from_df, resize_coordinates

sys.path.insert(0, f'{bpreveal_path}/src')
import losses

## Filesystem commands
!mkdir -p \
    json \
    bed/bpreveal \
    input \
    models \
    preds \
    scripts/modeling \
    tsv \
    shap \
    npz \
    {figure_path}

!mkdir {figure_path}

2026-02-04 10:25:26.187986: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-04 10:25:26.188025: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-04 10:25:26.189493: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-04 10:25:26.197041: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-04 10:25:30.310425: W tensorflow/comp

mkdir: cannot create directory ‘figures/3c_train_differentiation_combined’: File exists


## Custom functions

The following function takes a list of str commands and generates a SLURM batch job with parameter allocations. This is mostly specific to Stowers computational resources, would recommend reworking the job submissions to match your own institutional setup if reproducing.

In [3]:
def generate_slurm_array(cmds, output_file, simultaneous_jobs = 3, gpu = True, mem = 200):
    total_jobs = len(cmds)
    if gpu:
        gpu_fill = '#SBATCH --partition=gpu\n#SBATCH --gres=gpu:1\n#SBATCH --cpus-per-task=10'
    else:
        gpu_fill = '#SBATCH --cpus-per-task=10'
    array_header = ['#!/usr/bin/bash',
                  '#SBATCH --job-name bpnet_training',
                  '#SBATCH --output=slurm_%j.log',
                  f'#SBATCH --array=1-{total_jobs}%{simultaneous_jobs}',
                  f'#SBATCH --mem={mem}gb', 
                  '#SBATCH --time=72:00:00',
                  gpu_fill,
                  'source /home/mw2098/.bashrc',
                  'conda deactivate',
                  'conda activate bpreveal_404',
                  'ml bedtools',
                  f'cd {working_dir}']
    # array_cmds = array_header
    for i,cmd in enumerate(cmds):
        array_cmd = ["if [[ ${{SLURM_ARRAY_TASK_ID}} == {0:d} ]] ; then\n".format(i+1), 
                    "    {0:s}\n".format(cmd),
                    "fi\n\n"]
        array_header = array_header + array_cmd 
        
    with open(output_file, mode='wt') as slurm:
        slurm.write('\n'.join(array_header))
        slurm.write('\n')

# Set up static modeling parameters

Here are the variables that will remain static throughout the optimization process. 

In [4]:
# Default model architecture
output_length = 1000
max_jitter = 100
batch_size = 64
learning_rate = 0.004 #BPReveal is adaptive, so this learning rate doesn't need optimizing.
early_stopping_patience = 8
reverse_complement = True
seed = 2346
trials = 256

#Annotations
genome = '../0_setup/fa/mm10.fa'
test_chroms = ['chr1', 'chr8', 'chr17']
val_chroms = ['chr2', 'chr3', 'chr9']
train_chroms = ['chr4', 'chr5', 'chr6', 'chr7', 'chr10', 'chr11', 'chr12', 'chr13', 
                'chr14', 'chr15', 'chr16', 'chr18', 'chr19', 'chrX']
profile_metrics_of_interest = ['mnll', 'jsd']
counts_metrics_of_interest = ['counts-pearson', 'counts-spearman']
model_suffix = ['_residual.model', '_combined.model']

"""
Optimized architectures based on previous model training.
"""

train_settings_dict = {
    'bias_atac' : { #Generic Tn5 bias model
        'n_dil_layers' : 4,
        'conv1_kernel_size' : 21,
        'profile_kernel_size' : 75,    
        'filters' : 128,
        'counts_loss_weight' : 10,    
        'output_length': 1000,
    },
    'residual_atac' : { #Generic binding model
        'n_dil_layers' : 8,
        'conv1_kernel_size' : 7,
        'profile_kernel_size' : 7,
        'filters' : 128,
        'output_length': 1000,
        'counts_loss_weight' : 100
    }
}


#Calculate associate input lengths
input_length_dict = {}
for k,v in train_settings_dict.items():
    proxy_length = !{python_path} {bpreveal_path}/src/lengthCalc.py \
        --output-len {v['output_length']} \
        --n-dil-layers {v['n_dil_layers']} \
        --conv1-kernel-size {v['conv1_kernel_size']} \
        --profile-kernel-size {v['profile_kernel_size']}
    input_length_dict[k] = int(proxy_length[0])
input_length_dict

{'bias_atac': 1154, 'residual_atac': 2032}

In [5]:

#Second-pass models to train
modeling_design_dict = {    

    'bunina_atac_day0_across_combined': {
        'regions': {'bunina_day0': 'narrowpeak/bunina_atac_day0_peaks.narrowPeak',
                    'bunina_day4': 'narrowpeak/bunina_atac_day4_peaks.narrowPeak',
                    'bunina_day8': 'narrowpeak/bunina_atac_day8_peaks.narrowPeak',
                    'bunina_day12': 'narrowpeak/bunina_atac_day12_peaks.narrowPeak'},
        'cov': {'atac': {'cutsite': 'bw/Bunina_mesc_atac_day0_cutsites_combined.bw'}},
        'bias_model_prefix': 'atac_wt', 'bias_settings_name': 'bias_atac', 'use_bias_counts': False,
        'train_settings_name': 'residual_atac'
    },
    'bunina_atac_day4_across_combined': {
        'regions': {'bunina_day0': 'narrowpeak/bunina_atac_day0_peaks.narrowPeak',
                    'bunina_day4': 'narrowpeak/bunina_atac_day4_peaks.narrowPeak',
                    'bunina_day8': 'narrowpeak/bunina_atac_day8_peaks.narrowPeak',
                    'bunina_day12': 'narrowpeak/bunina_atac_day12_peaks.narrowPeak'},
        'cov': {'atac': {'cutsite': 'bw/Bunina_mesc_atac_day4_cutsites_combined.bw'}},
        'bias_model_prefix': 'atac_wt', 'bias_settings_name': 'bias_atac', 'use_bias_counts': False,
        'train_settings_name': 'residual_atac'
    },
    'bunina_atac_day8_across_combined': {
        'regions': {'bunina_day0': 'narrowpeak/bunina_atac_day0_peaks.narrowPeak',
                    'bunina_day4': 'narrowpeak/bunina_atac_day4_peaks.narrowPeak',
                    'bunina_day8': 'narrowpeak/bunina_atac_day8_peaks.narrowPeak',
                    'bunina_day12': 'narrowpeak/bunina_atac_day12_peaks.narrowPeak'},
        'cov': {'atac': {'cutsite': 'bw/Bunina_mesc_atac_day8_cutsites_combined.bw'}},
        'bias_model_prefix': 'atac_wt', 'bias_settings_name': 'bias_atac', 'use_bias_counts': False,
        'train_settings_name': 'residual_atac'
    },
    'bunina_atac_day12_across_combined': {
        'regions': {'bunina_day0': 'narrowpeak/bunina_atac_day0_peaks.narrowPeak',
                    'bunina_day4': 'narrowpeak/bunina_atac_day4_peaks.narrowPeak',
                    'bunina_day8': 'narrowpeak/bunina_atac_day8_peaks.narrowPeak',
                    'bunina_day12': 'narrowpeak/bunina_atac_day12_peaks.narrowPeak'},
        'cov': {'atac': {'cutsite': 'bw/Bunina_mesc_atac_day12_cutsites_combined.bw'}},
        'bias_model_prefix': 'atac_wt', 'bias_settings_name': 'bias_atac', 'use_bias_counts': False,
        'train_settings_name': 'residual_atac'
    },

}

## Modify `modeling_cov_dict` 

Create .json configurations for the different steps for BPReveal training. 

In [6]:
#Prepare .bw configurations. Remember, order matters here.
modeling_cov_dict = {}
for k,v in modeling_design_dict.items():
    bw_dict = {}
    if type(v['cov'])==str:
        bw_dict['prepare_bed_inputs'] = [{'bigwig-names': [v['cov']], 'max-quantile': 1, 'min-counts': 1}]
        bw_dict['prepare_train_inputs'] = [{'bigwig-files': [v['cov']], 'revcomp-task-order': 'auto'}]
    else:
        bw_dict['prepare_bed_inputs'] = [{'bigwig-names': list(v2.values()), 'max-quantile': 1, 'min-counts': 1} for v2 in v['cov'].values()]
        bw_dict['prepare_train_inputs'] = [{'bigwig-files': list(v2.values()), 'revcomp-task-order': 'auto'} for v2 in v['cov'].values()]
    modeling_cov_dict[k] = bw_dict
modeling_cov_dict.keys()

dict_keys(['bunina_atac_day0_across_combined', 'bunina_atac_day4_across_combined', 'bunina_atac_day8_across_combined', 'bunina_atac_day12_across_combined'])

## Prepare 3-fold region datasets

Define different chromosome folds for training.

In [7]:
chromosome_folds_dict = {
    'fold1': {'train' : train_chroms,
              'val' : val_chroms,
              'test' : test_chroms},
    'fold2': {'train' : ['chr1', 'chr2', 'chr6', 'chr7', 'chr8', 'chr9', 'chr11', 'chr12', 
                       'chr13', 'chr14', 'chr15','chr17', 'chr18', 'chr19'],
              'val' : ['chr3', 'chr5', 'chr16'],
              'test' : ['chr4', 'chr10', 'chrX']},
    'fold3': {'train' : ['chr2', 'chr3', 'chr6', 'chr8','chr9', 'chr10', 'chr11', 'chr12', 
                       'chr15', 'chr16','chr17', 'chr18', 'chr19', 'chrX'],
              'val' : ['chr1', 'chr7', 'chr13'],
              'test' : ['chr4', 'chr5', 'chr14']}
}

# Train, predict, assess and SHAP final models

After optimization, we have architectures that are well-suited for our different types of models and training sets. Thus, we can train each model set over three different folds of chromosomes to ensure that models are stable and consistently returning the same sequence grammar. 

## Define random sequences

In [8]:
#Calculate associate input lengths
for k,v in train_settings_dict.items():
    proxy_length = !{python_path} {bpreveal_path}/src/lengthCalc.py \
        --output-len {v['output_length']} \
        --n-dil-layers {v['n_dil_layers']} \
        --conv1-kernel-size {v['conv1_kernel_size']} \
        --profile-kernel-size {v['profile_kernel_size']}
    input_length_dict[k] = int(proxy_length[0])
input_length_dict

{'bias_atac': 1154, 'residual_atac': 2032}

In [9]:
for k,v in input_length_dict.items():
    random_seqs_path = f'npz/random_seqs_seed_{seed}_input_{v}_trials_{trials}_array.npz'
    seqs = [np.random.choice(['A', 'C','G', 'T'], v) for i in range(trials)]
    seqs_1he = one_hot_encode_sequences(seqs)
    np.savez(random_seqs_path, seqs_1he = seqs_1he)

## Prepare training datasets

First, write the .json files required to generate training datasets.

In [10]:
#Loop through different input_lengths based on the different model requirements to generate data.
for model_name,model_bws in modeling_cov_dict.items():
    for fold_name,fold_chroms in chromosome_folds_dict.items():        
        fold_prefix = model_name + '_' + str(fold_name)
        input_length = input_length_dict[modeling_design_dict[model_name]['train_settings_name']]
        output_length = train_settings_dict[modeling_design_dict[model_name]['train_settings_name']]['output_length']
        max_jitter = int(output_length*.2)
    
        #Consolidate into a .json-readable dictionary
        prepare_bed_regions_dict = {'heads': model_bws['prepare_bed_inputs'],
                                    'splits': {"test-chroms": fold_chroms['test'], 
                                               "val-chroms": fold_chroms['val'],
                                               "train-chroms": fold_chroms['train'],
                                               "regions": list(modeling_design_dict[model_name]['regions'].values())},
                                    'genome': f'{genome}',
                                    'write-counts-to': f'bed/bpreveal/{fold_prefix}_all.stats',
                                    'output-length': output_length,
                                    'input-length': input_length,
                                    'max-jitter': max_jitter,
                                    'output-prefix': f'bed/bpreveal/{fold_prefix}',
                                    'resize-mode': 'center',
                                    'remove-overlaps': False,
                                    'verbosity': 'INFO'} #Switch this to DEBUG if you are missing regions
        prepare_bed_regions_json = json.dumps(prepare_bed_regions_dict, indent=4)
        bed_json_file = f'json/prepareBedPeaks_{fold_prefix}.json'
        with open(bed_json_file, 'w') as outfile:
            outfile.write(prepare_bed_regions_json)

        # Prepare input training data
        prepare_input_train_dict = {'genome': genome, 
                            'input-length': input_length, 
                            'output-length': output_length, 
                            'max-jitter': max_jitter, 
                            'regions': f'bed/bpreveal/{fold_prefix}_train.bed', 
                            'output-h5': f'input/{fold_prefix}_train.h5', 
                            'heads': model_bws['prepare_train_inputs'], 
                            'reverse-complement': reverse_complement,
                            'verbosity': 'DEBUG'} 
        prepare_input_train_json = json.dumps(prepare_input_train_dict, indent=4)
        train_json_file = f'json/prepareInputTrain_{fold_prefix}.json'        
        with open(train_json_file, 'w') as outfile:
            outfile.write(prepare_input_train_json)
            
        # Prepare validation training data
        prepare_input_valid_dict = {'genome': genome, 
                    'input-length': input_length, 
                    'output-length': output_length, 
                    'max-jitter': max_jitter, 
                    'regions': f'bed/bpreveal/{fold_prefix}_val.bed', 
                    'output-h5': f'input/{fold_prefix}_val.h5', 
                    'heads': model_bws['prepare_train_inputs'], 
                    'reverse-complement': reverse_complement,
                    'verbosity': 'DEBUG'} 
        prepare_input_valid_json = json.dumps(prepare_input_valid_dict, indent=4)
        valid_json_file = f'json/prepareInputValid_{fold_prefix}.json'        
        with open(valid_json_file, 'w') as outfile:
            outfile.write(prepare_input_valid_json)

Next, generate SLURM jobs to generate the .h5 and .bed files required for training/validation datasets.

In [11]:
array_cmds = []
for model_name,model_bws in modeling_cov_dict.items():
    for fold_name,fold_chroms in chromosome_folds_dict.items():        
        fold_prefix = model_name + '_' + str(fold_name)
        
        prepare_peaks_cmd = f'{python_path} {bpreveal_path}/src/prepareBed.py json/prepareBedPeaks_{fold_prefix}.json'
        prepare_train_cmd = f'{python_path} {bpreveal_path}/src/prepareTrainingData.py json/prepareInputTrain_{fold_prefix}.json'
        prepare_valid_cmd = f'{python_path} {bpreveal_path}/src/prepareTrainingData.py json/prepareInputValid_{fold_prefix}.json'

        #Write to a job array
        cmds_str = '\n'.join([prepare_peaks_cmd] + [prepare_train_cmd] + [prepare_valid_cmd])
        array_cmd = [cmds_str]
        array_cmds += array_cmd

generate_slurm_array(cmds = array_cmds, output_file = 'scripts/modeling/bpnet_residual_models_prepare_data.slurm',
                     simultaneous_jobs = 20, gpu = False)
print('sbatch scripts/modeling/bpnet_residual_models_prepare_data.slurm')

sbatch scripts/modeling/bpnet_residual_models_prepare_data.slurm


## Generate training scripts

Here, we will generate training .json configuration files and training scripts for the final training.

In [12]:
array_cmds = []
for model_name,model_info in modeling_design_dict.items():
    tasks = list(model_info['cov'].keys())
    num_channels = len(list(model_info['cov'].values())[0].keys())
    train_settings = train_settings_dict[model_info['train_settings_name']]
    input_length = input_length_dict[model_info['train_settings_name']]
                       
    for fold_name,fold_chroms in chromosome_folds_dict.items():
        
        fold_prefix = model_name + '_' + str(fold_name)

        #Determine bias/transform model information for training combined model.
        bias_prefix = os.path.basename(model_info['bias_model_prefix']) # + '_fold1'
        trans_prefix = bias_prefix + '_trans'

        #Set up training parameters for the base model
        train_dict = {'settings': {'output-prefix': f'models/{fold_prefix}', 
                                   'transformation-model': {'transformation-model-file' : f'models/{trans_prefix}.model'},
                                   'epochs': 200, 
                                   'early-stopping-patience': early_stopping_patience, 
                                   'batch-size': batch_size, 
                                   'learning-rate': learning_rate, 
                                   'learning-rate-plateau-patience': 5, 
                                   'max-jitter': max_jitter, 
                                   'architecture': {'architecture-name': 'bpnet', 
                                                    'input-length': input_length, 
                                                    'output-length': output_length, 
                                                    'model-name': fold_prefix, 
                                                    'model-args': '', 
                                                    'filters': train_settings['filters'], 
                                                    'layers': train_settings['n_dil_layers'], 
                                                    'input-filter-width': train_settings['conv1_kernel_size'], 
                                                    'output-filter-width': train_settings['profile_kernel_size']}}, 
                      'heads': [{'num-tasks': num_channels, 
                             'profile-loss-weight': 1, 
                             'head-name': task, 
                             'counts-loss-frac-target': .1,
                                     'counts-loss-weight': train_settings['counts_loss_weight'],
                                     'use-bias-counts': model_info['use_bias_counts']} for task in tasks],
                      'train-data': f'input/{fold_prefix}_train.h5', 
                      'val-data': f'input/{fold_prefix}_val.h5', 
                      'verbosity': 'WARNING'}

        train_json = json.dumps(train_dict, indent=4)
        train_json_file = f'json/trainCombinedModel_{fold_prefix}.json'
        with open(train_json_file, 'w') as outfile:
            outfile.write(train_json)

        #Set up prediction parameters, predict both combined and residual models
        model_predict = []
        for ms in model_suffix:
            ms_short = ms.replace('.model', '')
            pred_dict = {'settings': {'genome': genome, 
                                      'output-h5': f'preds/{fold_prefix}{ms_short}.h5', 
                                      'batch-size': batch_size, 
                                      'heads': len(tasks), 
                                      'architecture': {'model-file': f'models/{fold_prefix}{ms}', 
                                                       'input-length': input_length, 
                                                       'output-length': output_length}}, 
                         'bed-file': f'bed/bpreveal/{fold_prefix}_all.bed',
                         # 'bed-file':f'bed/bpreveal/atac_wt_fold1_all.bed', 
                         'verbosity': 'DEBUG'}
            pred_json = json.dumps(pred_dict, indent=4)
            pred_json_file = f'json/predictPeaks_{fold_prefix}{ms_short}.json'
            with open(pred_json_file, 'w') as outfile:
                outfile.write(pred_json)

            cmd = f'{python_path} {bpreveal_path}/src/makePredictionsBed.py json/predictPeaks_{fold_prefix}{ms_short}.json'
            model_predict.append(cmd)
        
        #Set up training and prediction commands
        model_train = [f'{python_path} {bpreveal_path}/src/trainCombinedModel.py json/trainCombinedModel_{fold_prefix}.json']
            
        #Write to a job array
        cmds_str = '\n'.join(model_train + model_predict)
        # cmds_str = '\n'.join(model_predict)
        array_cmd = [cmds_str]
        array_cmds += array_cmd

generate_slurm_array(cmds = array_cmds, output_file = 'scripts/modeling/bpnet_residual_models_train.slurm', mem = 500)
print('sbatch scripts/modeling/bpnet_residual_models_train.slurm')

sbatch scripts/modeling/bpnet_residual_models_train.slurm


## Generate prediction .bw files and assess performance

Here, we will extract .bw coverage from the generated .h5 files and compare them to generated features in order to compute performance metrics

In [13]:
array_cmds = []
for model_name,model_info in modeling_design_dict.items():               
    for fold_name,fold_chroms in chromosome_folds_dict.items():
        fold_prefix = model_name + '_' + str(fold_name)
        for head_counter, (task, cov) in enumerate(model_info['cov'].items()):
            for channel_counter, (channel_name, channel) in enumerate(cov.items()):
                for ms in model_suffix:
                    ms_short = ms.replace('.model', '')
                    predict_cmd = f'{python_path} {bpreveal_path}/src/predictToBigwig.py \
                    --h5 preds/{fold_prefix}{ms_short}.h5 \
                    --bw preds/{fold_prefix}_{task}_{channel_name}{ms_short}.bw \
                    --head-id {head_counter} \
                    --task-id {channel_counter} --mode profile --verbose'
    
                    #Write to a job array
                    cmds_str = '\n'.join([predict_cmd])
                    array_cmd = [cmds_str]
                    array_cmds += array_cmd

generate_slurm_array(cmds = array_cmds, output_file = 'scripts/modeling/bpnet_residual_models_predict.slurm',
                     simultaneous_jobs = 20, gpu = False)
print('sbatch scripts/modeling/bpnet_residual_models_predict.slurm')

sbatch scripts/modeling/bpnet_residual_models_predict.slurm


## Compute performance metrics

Here, we will compute and compare performance metrics on the test set of chromosomes. We will assess the 50th percentile as the key feature of profile metrics as well as the overall total counts performance. 

Note that we will also calculate the first-pass model performances across the same regions, such that we might fairly assess whether first- or the corralled second-pass combined models more rigorously learn the information. 

First, calculate performance metrics and save in a .json file. NOTE: Here, we will be using the referenced model in 3_train_models.ipynb in order to do a fair comparison. Predicting no accessibility across the entire genome is NOT the same as learning thoughtful regulatory grammar.

In [14]:
#Calculate residual modelshe e
array_cmds = []
for model_name,model_info in modeling_design_dict.items():
    for fold_name,fold_chroms in chromosome_folds_dict.items():
        fold_prefix = model_name + '_' + str(fold_name)
        for dataset in ['test','val','train']:
            peaks_path = f'bed/bpreveal/{fold_prefix}_{dataset}.bed'
            # peaks_path = f'bed/bpreveal/atac_wt_fold1_all.bed'
            
            for head_counter, (task, cov) in enumerate(model_info['cov'].items()):
                for channel_counter, (channel_name, channel) in enumerate(cov.items()):
                    for ms in model_suffix:
                        ms_short = ms.replace('.model', '')    
                        reference_path = channel
                        preds_path = f'preds/{fold_prefix}_{task}_{channel_name}{ms_short}.bw'
                        
                        #Compute correlations and profile metrics over quantiles
                        metrics_json_file = f'json/metrics_{fold_prefix}_{task}_{channel_name}{ms_short}_{dataset}.json'

                        metrics_cmd = f'{python_path} {bpreveal_path}/src/metrics.py \
                            --reference {reference_path} \
                            --pred {preds_path} \
                            --regions {peaks_path} \
                            --threads 70 --json-output --apply-abs --skip-zeroes | tee {metrics_json_file}'
                        
                        #Write to a job array
                        cmds_str = '\n'.join([metrics_cmd])
                        array_cmd = [cmds_str]
                        array_cmds += array_cmd

             
generate_slurm_array(cmds = array_cmds, output_file = 'scripts/modeling/bpnet_residual_models_metrics.slurm',
                     simultaneous_jobs = 20, gpu = False)
print('sbatch scripts/modeling/bpnet_residual_models_metrics.slurm')

sbatch scripts/modeling/bpnet_residual_models_metrics.slurm


Next, consolidate performance metrics by the different parameters and save into a tidy .tsv file. 

In [15]:
metrics_df = pd.DataFrame()
for model_name,model_info in modeling_design_dict.items():
    for fold_name,fold_chroms in chromosome_folds_dict.items():
        fold_prefix = model_name + '_' + str(fold_name)
        
        #Determine bias/transform model information for training combined model.
        bias_prefix = os.path.basename(model_info['bias_model_prefix']) #+ '_' + str(fold_name)
        
        for dataset in ['test','val','train']:
            peaks_path = f'bed/bpreveal/{fold_prefix}_{dataset}.bed'
            for head_counter, (task, cov) in enumerate(model_info['cov'].items()):
                for channel_counter, (channel_name, channel) in enumerate(cov.items()):

                    for ms in model_suffix:
                        ms_short = ms.replace('.model', '')    
                        reference_path = channel
                        preds_path = f'preds/{fold_prefix}_{task}_{channel_name}{ms_short}.bw'
                        
                        #Compute correlations and profile metrics over quantiles
                        metrics_json_file = f'json/metrics_{fold_prefix}_{task}_{channel_name}{ms_short}_{dataset}.json'
                        
                        with open(metrics_json_file) as f:
                            metrics_dict = json.load(f)
    
                        #Format into a tidy df
                        df = pd.DataFrame([os.path.basename(metrics_dict['predicted'])]).transpose()
                        df.columns = ['bw']
                        df[['model', 'bw']] = df['bw'].str.split('_fold', n=1, expand=True)
                        df[['fold', 'bw']] = df['bw'].str.split('_', n=1, expand=True)
                        df[['task', 'bw']] = df['bw'].str.split('_', n=1, expand=True)
                        df[['channel', 'bw']] = df['bw'].str.split('_', n=1, expand=True)
                        df[['filler', 'bw']] = df['bw'].str.split(f'.bw', n=1, expand=True)
                        df['dataset'] = dataset
                        df = df.drop('bw', axis = 1)
                        df[profile_metrics_of_interest] = [np.median(metrics_dict[metric]['quantiles']) for metric in profile_metrics_of_interest]
                        df[counts_metrics_of_interest] = [metrics_dict[metric] for metric in counts_metrics_of_interest]
                        metrics_df = pd.concat([metrics_df, df])

metrics_df.to_csv('tsv/performance_metrics_across_differentiation_residual_train.tsv.gz', sep = '\t', index = False)

In [16]:
metrics_df = pd.read_csv('tsv/performance_metrics_across_differentiation_residual_train.tsv.gz', sep = '\t')
metrics_df[metrics_df.dataset=='test']

,model,fold,task,channel,filler,dataset,mnll,jsd,counts-pearson,counts-spearman
0,bunina_atac_day0_across_combined,1,atac,cutsite,residual,test,-657.776208,0.657231,0.737655,0.789876
1,bunina_atac_day0_across_combined,1,atac,cutsite,combined,test,-439.586305,0.501861,0.737702,0.789918
6,bunina_atac_day0_across_combined,2,atac,cutsite,residual,test,-603.976970,0.664208,0.692151,0.764370
7,bunina_atac_day0_across_combined,2,atac,cutsite,combined,test,-413.797314,0.512108,0.692329,0.764499
12,bunina_atac_day0_across_combined,3,atac,cutsite,residual,test,-635.937523,0.657743,0.768343,0.776299
13,bunina_atac_day0_across_combined,3,atac,cutsite,combined,test,-434.125316,0.504518,0.768462,0.776428
18,bunina_atac_day4_across_combined,1,atac,cutsite,residual,test,-731.094592,0.643199,0.853739,0.897825
19,bunina_atac_day4_across_combined,1,atac,cutsite,combined,test,-484.189818,0.482506,0.853803,0.897824
24,bunina_atac_day4_across_combined,2,atac,cutsite,residual,test,-671.626312,0.658574,0.796089,0.880788
25,bunina_atac_day4_across_combined,2,atac,cutsite,combined,test,-446.176521,0.501893,0.796210,0.880870


These performance metrics will be more directly compared in the next .Rmd. 

## Calculate SHAP scores for all models

### Generate SHAP scores

Prepare SHAP regions. 

In [17]:
array_cmds = []

for model_name,model_info in modeling_design_dict.items():
    num_channels = len(list(model_info['cov'].values())[0].keys())
    input_length = input_length_dict[model_info['train_settings_name']]
    
    for fold_name,fold_chroms in chromosome_folds_dict.items():
        fold_prefix = model_name + '_' + str(fold_name)

        #Set up SHAP parameters for each task.
        shap_template_dict = {
            'genome': genome,
            'bed-file': f'bed/bpreveal/{fold_prefix}_all.bed',
            'input-length': input_length,
            'output-length': output_length,
            'num-shuffles': 20,
            'verbosity': 'DEBUG'
        }
        
        shap_dict = shap_template_dict.copy()
        shap_dict['model-file'] = f'models/{fold_prefix}_residual.model'
        for head_counter, (task, cov) in enumerate(model_info['cov'].items()):
            shap_dict['heads'] = len(list(model_info['cov'].keys()))
            shap_dict['head-id'] = head_counter
            shap_dict['profile-task-ids'] = list(range(num_channels))
            shap_dict['profile-h5'] = f'shap/{fold_prefix}_{task}_profile.h5'
            shap_dict['counts-h5'] = f'shap/{fold_prefix}_{task}_counts.h5'
            shap_json = json.dumps(shap_dict, indent=4)
            shap_json_file = f'json/shapFlat_{fold_prefix}_{task}.json'
            with open(shap_json_file, 'w') as outfile:
                outfile.write(shap_json)   

            #Set up the SHAP commands
            model_shap = [f'{python_path} {bpreveal_path}/src/interpretFlat.py json/shapFlat_{fold_prefix}_{task}.json']
                
            #Write to a job array
            cmds_str = '\n'.join(model_shap)
            array_cmd = [cmds_str]
            array_cmds += array_cmd

generate_slurm_array(cmds = array_cmds, output_file = 'scripts/modeling/bpnet_residual_models_shap.slurm')
print('sbatch scripts/modeling/bpnet_residual_models_shap.slurm')

sbatch scripts/modeling/bpnet_residual_models_shap.slurm


### Convert SHAP .h5 to .bw

In [ ]:
array_cmds = []
for model_name,model_info in modeling_design_dict.items():
    for fold_name,fold_chroms in chromosome_folds_dict.items():
        fold_prefix = model_name + '_' + str(fold_name)
        for task in list(model_info['cov'].keys()):

            counts_shap_cmd = f'{python_path} {bpreveal_path}/src/shapToBigwig.py \
            --h5 shap/{fold_prefix}_{task}_counts.h5 \
            --bw shap/{fold_prefix}_{task}_counts.bw \
            --verbose'

            profile_shap_cmd = f'{python_path} {bpreveal_path}/src/shapToBigwig.py \
            --h5 shap/{fold_prefix}_{task}_profile.h5 \
            --bw shap/{fold_prefix}_{task}_profile.bw \
            --verbose'

            #Write to a job array
            cmds_str = '\n'.join([profile_shap_cmd] + [counts_shap_cmd])
            array_cmd = [cmds_str]
            array_cmds += array_cmd

generate_slurm_array(cmds = array_cmds, output_file = 'scripts/modeling/bpnet_residual_models_shap_to_bw.slurm',
                     simultaneous_jobs = 20, gpu = False, mem = 20)
print('sbatch scripts/modeling/bpnet_residual_models_shap_to_bw.slurm')